<a href="https://colab.research.google.com/github/MuhammadSarimUmer/Python-Learning/blob/master/Echo_Mental_Health_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌿 Echo — Mental Health Companion
### A compassionate AI for mental wellness

**Features:**
- 🤖 Groq-powered LLM chat (llama3-70b)
- 🔊 Text-to-Speech (gTTS)
- 📚 RAG system (FAISS + HuggingFace embeddings)
- 🎭 Mood buttons with AI responses
- 🚨 Crisis detection + instant help
- 🧘 Built-in coping tools
- 📞 Emergency resources
- 💾 Local JSON chat storage + delete
- 🪟 Context window management
- 🌐 Gradio glassmorphism UI

**Run each cell in order!**

## 📦 Cell 1 — Install Dependencies

In [1]:
%%capture
!pip install groq \
    'langchain>=0.2,<0.4' \
    langchain-community \
    langchain-groq \
    langchain-text-splitters \
    faiss-cpu \
    sentence-transformers \
    gtts \
    'gradio>=4.44,<6.0' \
    pydub numpy


## 🔑 Cell 2 — Set Your Groq API Key

In [21]:
import os
from google.colab import userdata

# Get API key from Colab Secrets
GROQ_API_KEY = userdata.get("GROQ_API_KEY")  # or "GROQ" depending on what you want

if not GROQ_API_KEY:
    raise ValueError("❌ GROQ API key not found in Colab Secrets")

# Clean it (important to avoid hidden unicode issues)
GROQ_API_KEY = GROQ_API_KEY.strip().replace("\n", "").replace("\r", "")

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

print("✅ API key loaded from Colab Secrets successfully!")

✅ API key loaded from Colab Secrets successfully!


## 🧠 Cell 3 — Core Backend (RAG, LLM, TTS, Crisis Detection)

In [25]:
import os, json, re, datetime, tempfile, unicodedata
from pathlib import Path
from gtts import gTTS
from groq import Groq
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.docstore.document import Document

# -------------------------------
# ✅ SAFE TEXT NORMALIZATION (FIX ONLY)
# -------------------------------
def normalize_text(text):
    if not isinstance(text, str):
        return ""

    text = unicodedata.normalize("NFKC", text)

    replacements = {
        "—": "-",
        "–": "-",
        "“": '"',
        "”": '"',
        "’": "'",
        "…": "...",
    }

    for k, v in replacements.items():
        text = text.replace(k, v)

    return text.strip()


# 1. KNOWLEDGE BASE (UNCHANGED)
MENTAL_HEALTH_DOCS = [
    """Anxiety is a natural response to stress. Common symptoms include racing heart, sweating, trembling, shortness of breath, and persistent worry.
    Cognitive Behavioral Therapy (CBT) is highly effective for anxiety. It helps identify and challenge distorted thought patterns.
    Grounding techniques like the 5-4-3-2-1 method can interrupt anxiety spirals: name 5 things you see, 4 you can touch, 3 you hear, 2 you smell, 1 you taste.
    Regular exercise, especially aerobic activity for 30 minutes 3-5 times a week, significantly reduces anxiety symptoms.
    Limiting caffeine and alcohol can reduce anxiety. Both substances can trigger or worsen symptoms.""",

    """Depression involves persistent sadness, loss of interest, changes in sleep and appetite, fatigue, and feelings of worthlessness.
    Behavioral activation means scheduling small enjoyable activities and is a core depression treatment.
    Sleep hygiene is critical. Consistent sleep and wake times and limiting screens before bed improve mood significantly.
    Social connection is protective. Isolation worsens depression. Even brief social contact helps.
    Self-compassion involves treating yourself with the same kindness you would offer a good friend.""",

    """Breathing exercises activate the parasympathetic nervous system and reduce stress hormones within minutes.
    Box breathing: inhale for 4 counts, hold for 4, exhale for 4, hold for 4. Repeat 4-6 times.
    4-7-8 breathing: inhale for 4 counts, hold for 7, exhale slowly for 8. This promotes relaxation and can aid sleep.
    Diaphragmatic breathing: breathe into your belly, not chest. Only the belly hand should rise.
    Alternate nostril breathing from yoga reduces stress and improves focus.""",

    """Mindfulness is the practice of non-judgmental present-moment awareness. It reduces rumination, stress, and emotional reactivity.
    Body scan meditation: slowly move attention from feet to head, noticing sensations without judgment. Takes 5-20 minutes.
    MBSR (Mindfulness-Based Stress Reduction) is an 8-week program proven to reduce anxiety, depression, and chronic pain.
    Mindful walking: walk slowly, feel each step, notice your breath and surroundings.
    Journaling 3 things you are grateful for daily rewires the brain toward positivity and reduces depression over time.""",

    """Crisis resources: If you are in immediate danger, call emergency services (911 in USA, 115 in Pakistan, 999 in UK).
    Pakistan crisis line: Umang helpline 0317-4288665, available Monday-Saturday 3pm-9pm.
    iCall India: 9152987821. Vandrevala Foundation 24/7: 1860-2662-345.
    Samaritans UK: 116 123 (free, 24/7). Crisis Text Line US: Text HOME to 741741.
    Reaching out for help is a sign of strength, not weakness. You are not alone.""",

    """Panic attacks are intense surges of fear that peak within minutes. They are not dangerous but feel terrifying.
    During a panic attack: remind yourself this will pass, I am safe, this is temporary.
    Cold water on the face triggers the diving reflex and rapidly slows heart rate.
    Progressive muscle relaxation: systematically tense and release muscle groups from toes to face.
    Instead of fighting panic, accepting it reduces its power.""",

    """Stress management involves identifying stressors, building coping resources, and creating recovery time.
    Time management techniques like the Pomodoro method (25 min focus, 5 min break) reduce overwhelm.
    Setting boundaries at work and in relationships protects mental energy. Saying no is self-care.
    Nature exposure (even 20 minutes) reduces cortisol.
    Creative expression such as art, music, and writing provides emotional release and reduces stress hormones.""",

    """Self-care is not selfish. It includes physical, emotional, social, spiritual, and professional dimensions.
    Physical self-care: regular sleep, nutrition, movement, medical checkups, limiting substances.
    Emotional self-care: therapy, journaling, processing emotions, setting limits on news and social media.
    Social self-care: nurturing supportive relationships, spending time with loved ones, seeking community.
    Professional help: therapists, counselors, and psychiatrists provide evidence-based treatment. There is no shame in seeking help."""
]

# 2. BUILD RAG (UNCHANGED LOGIC)
print("Building RAG knowledge base...")
splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=60)
docs = []
for text in MENTAL_HEALTH_DOCS:
    chunks = splitter.split_text(text)
    docs.extend([Document(page_content=c) for c in chunks])

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)
vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("RAG ready!")

# 3. GROQ CLIENT (UNCHANGED)
client = Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL = "llama-3.1-8b-instant"

SYSTEM_PROMPT = """Your original prompt unchanged"""

# 4. CRISIS DETECTION (UNCHANGED)
CRISIS_KEYWORDS = [
    r"\bsuicid", r"\bkill myself\b", r"\bend my life\b", r"\bwant to die\b",
    r"\bno reason to live\b", r"\bhurt myself\b", r"\bself.harm",
    r"\bcant go on\b", r"\bgive up on life\b", r"\bnot worth living\b",
    r"\bwish i was dead\b", r"\bdie by suicide\b"
]

def detect_crisis(text):
    return any(re.search(kw, text.lower()) for kw in CRISIS_KEYWORDS)

# 5. CHAT STORAGE (FIXED UTF-8 ONLY)
CHAT_FILE = "Echo_chat_history.json"

def load_history():
    if Path(CHAT_FILE).exists():
        try:
            with open(CHAT_FILE, "r", encoding="utf-8") as f:
                return json.load(f)
        except:
            return []
    return []

def save_history(history):
    try:
        with open(CHAT_FILE, "w", encoding="utf-8") as f:
            json.dump(history, f, indent=2, ensure_ascii=False)
    except:
        pass

def delete_history():
    if Path(CHAT_FILE).exists():
        os.remove(CHAT_FILE)
    return []

# 6. TTS (FIX ONLY)
def text_to_speech(text, lang="en"):
    try:
        clean = normalize_text(text)
        clean = re.sub(r'\s+', ' ', clean).strip()[:800]

        tts = gTTS(text=clean, lang=lang, slow=False)
        tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3")
        tts.save(tmp.name)
        return tmp.name
    except Exception as e:
        print(f"TTS error: {e}")
        return None

# 7. RAG RETRIEVAL (UNCHANGED)
def retrieve_context(query):
    results = retriever.invoke(query)
    return "\n\n".join([d.page_content for d in results])

# 8. MAIN CHAT (FIX ONLY)
MAX_CONTEXT_TURNS = 8

def chat_with_Echo(user_message, conversation_history, enable_rag=True):
    user_message = normalize_text(user_message)

    is_crisis = detect_crisis(user_message)
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    if enable_rag:
        ctx = retrieve_context(user_message)
        if ctx:
            messages.append({"role": "system", "content": normalize_text("Relevant knowledge:\n" + ctx)})

    if is_crisis:
        messages.append({"role": "system", "content": "Crisis detected. Respond safely."})

    for turn in conversation_history[-(MAX_CONTEXT_TURNS * 2):]:
        messages.append({"role": turn["role"], "content": normalize_text(turn["content"])})

    messages.append({"role": "user", "content": user_message})

    response = client.chat.completions.create(
        model=MODEL, messages=messages, max_tokens=600, temperature=0.75
    )

    reply = normalize_text(response.choices[0].message.content)

    conversation_history.append({"role": "user", "content": user_message})
    conversation_history.append({"role": "assistant", "content": reply})
    save_history(conversation_history)

    return reply, text_to_speech(reply), conversation_history, is_crisis

print("✅ Backend ready (ONLY bug fixed, no features removed)")

Building RAG knowledge base...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RAG ready!
✅ Backend ready (ONLY bug fixed, no features removed)


## 🎨 Cell 4 — Gradio UI (Glassmorphism Design)

In [26]:
import gradio as gr

CUSTOM_CSS = """
:root {
    --bg-from: #0f0c29;
    --bg-mid: #302b63;
    --bg-to: #24243e;
    --glass: rgba(255,255,255,0.06);
    --glass-border: rgba(255,255,255,0.12);
    --glass-hover: rgba(255,255,255,0.10);
    --text-primary: #f0eeff;
    --text-secondary: rgba(240,238,255,0.65);
    --accent: #a78bfa;
    --accent-2: #34d399;
    --crisis: #f87171;
    --radius: 16px;
}
body, .gradio-container {
    background: linear-gradient(135deg, var(--bg-from) 0%, var(--bg-mid) 50%, var(--bg-to) 100%) !important;
    color: var(--text-primary) !important;
}
.gradio-container { max-width: 980px !important; margin: 0 auto !important; padding: 24px !important; }
.chatbot-wrap { background: transparent !important; border: 1px solid var(--glass-border) !important; border-radius: var(--radius) !important; }
textarea, input[type="text"] {
    background: rgba(255,255,255,0.05) !important;
    border: 1px solid var(--glass-border) !important;
    color: var(--text-primary) !important;
    border-radius: 12px !important;
}
.output-text {
    background: var(--glass) !important;
    border: 1px solid var(--glass-border) !important;
    border-radius: var(--radius) !important;
    padding: 16px !important;
    color: var(--text-primary) !important;
    line-height: 1.7 !important;
}
audio { width: 100% !important; border-radius: 10px !important; }
"""

def format_chat_for_gradio(history):
    messages = []
    i = 0
    while i < len(history) - 1:
        if history[i]["role"] == "user" and history[i+1]["role"] == "assistant":
            messages.append({"role": "user", "content": history[i]["content"]})
            messages.append({"role": "assistant", "content": history[i+1]["content"]})
            i += 2
        else:
            i += 1
    return messages

def on_send(message, history_state, enable_tts):
    if not message.strip():
        return history_state, format_chat_for_gradio(history_state), None, ""
    reply, audio_path, updated_history, is_crisis = chat_with_Echo(message, history_state)
    crisis_msg = ""
    if is_crisis:
        crisis_msg = "Crisis support activated. Please reach out to a professional. Pakistan: Umang 0317-4288665 | US: Text HOME to 741741"
    return updated_history, format_chat_for_gradio(updated_history), (audio_path if enable_tts else None), crisis_msg

def on_mood(mood, history_state, enable_tts):
    reply, audio_path, updated_history = handle_mood(mood, history_state)
    return updated_history, format_chat_for_gradio(updated_history), (audio_path if enable_tts else None), reply

def on_coping_tool(tool_name, enable_tts):
    tool = COPING_TOOLS.get(tool_name)
    if not tool:
        return "Tool not found.", None
    return tool["text"], (text_to_speech(tool["tts"]) if enable_tts else None)

def on_resources_tts(enable_tts):
    return text_to_speech(RESOURCES_TTS) if enable_tts else None

def on_delete_chat():
    history = delete_history()
    return history, [], None, "Chat history cleared."

def on_load():
    history = load_history()
    return history, format_chat_for_gradio(history)

with gr.Blocks(css=CUSTOM_CSS, title="Echo") as app:
    history_state = gr.State([])

    gr.HTML("""
    <div style="text-align:center;padding:32px 0 20px">
        <div style="font-size:2.5rem">🌿</div>
        <div style="font-size:2.4rem;font-weight:600;background:linear-gradient(135deg,#c4b5fd,#a78bfa,#34d399);-webkit-background-clip:text;-webkit-text-fill-color:transparent;background-clip:text">Echo</div>
        <div style="color:rgba(240,238,255,0.65);font-size:0.9rem;margin-top:6px">Your compassionate mental wellness companion</div>
        <div style="color:rgba(240,238,255,0.35);font-size:0.75rem;margin-top:8px;font-style:italic">Not a substitute for professional care. If in crisis, contact a professional immediately.</div>
    </div>
    """)

    with gr.Tabs():

        with gr.Tab("Chat with Echo"):
            with gr.Row():
                with gr.Column(scale=3):
                    chatbot = gr.Chatbot(
                        label="",
                        height=440,
                        elem_classes=["chatbot-wrap"],
                        type="messages"
                    )
                    crisis_banner = gr.Markdown("")
                    with gr.Row():
                        msg_input = gr.Textbox(
                            placeholder="Share what's on your mind... I'm here to listen",
                            label="",
                            lines=2,
                            scale=5
                        )
                        send_btn = gr.Button("Send", scale=1, variant="primary")
                    with gr.Row():
                        tts_toggle = gr.Checkbox(label="Enable voice responses", value=False)
                    audio_out = gr.Audio(label="Echo's Voice", autoplay=True)

                with gr.Column(scale=1):
                    gr.Markdown("**How are you feeling?**")
                    mood_output = gr.Markdown("", elem_classes=["output-text"])
                    moods = list(MOOD_PROMPTS.keys())
                    for i in range(0, len(moods), 2):
                        with gr.Row():
                            b1 = gr.Button(moods[i], size="sm")
                            b1.click(fn=on_mood, inputs=[gr.State(moods[i]), history_state, tts_toggle],
                                     outputs=[history_state, chatbot, audio_out, mood_output])
                            if i+1 < len(moods):
                                b2 = gr.Button(moods[i+1], size="sm")
                                b2.click(fn=on_mood, inputs=[gr.State(moods[i+1]), history_state, tts_toggle],
                                         outputs=[history_state, chatbot, audio_out, mood_output])
                    gr.Markdown("---")
                    delete_btn = gr.Button("Delete Chat History", variant="stop", size="sm")

            send_btn.click(
                fn=on_send,
                inputs=[msg_input, history_state, tts_toggle],
                outputs=[history_state, chatbot, audio_out, crisis_banner]
            ).then(lambda: "", outputs=[msg_input])

            msg_input.submit(
                fn=on_send,
                inputs=[msg_input, history_state, tts_toggle],
                outputs=[history_state, chatbot, audio_out, crisis_banner]
            ).then(lambda: "", outputs=[msg_input])

            delete_btn.click(fn=on_delete_chat, outputs=[history_state, chatbot, audio_out, crisis_banner])

        with gr.Tab("Coping Tools"):
            coping_tts_toggle = gr.Checkbox(label="Enable voice guidance", value=False)
            with gr.Row():
                with gr.Column(scale=3):
                    coping_output = gr.Markdown("Select a tool below to begin.", elem_classes=["output-text"])
                with gr.Column(scale=2):
                    coping_audio = gr.Audio(label="Guided Voice", autoplay=True)
            gr.Markdown("---")
            with gr.Row():
                for tool_name in COPING_TOOLS.keys():
                    btn = gr.Button(tool_name)
                    btn.click(fn=on_coping_tool, inputs=[gr.State(tool_name), coping_tts_toggle],
                              outputs=[coping_output, coping_audio])

        with gr.Tab("Resources"):
            with gr.Row():
                with gr.Column(scale=3):
                    gr.Markdown(RESOURCES_TEXT, elem_classes=["output-text"])
                with gr.Column(scale=2):
                    res_tts_toggle = gr.Checkbox(label="Enable voice", value=False)
                    read_btn = gr.Button("Read Aloud", variant="primary")
                    resources_audio = gr.Audio(label="Resources Audio", autoplay=True)
            read_btn.click(fn=on_resources_tts, inputs=[res_tts_toggle], outputs=[resources_audio])

        with gr.Tab("About"):
            gr.Markdown("""
## About Echo

Echo is a compassionate AI mental health companion.

| Feature | Technology |
|---------|------------|
| LLM Chat | Groq API (llama3-70b-8192) |
| Knowledge RAG | FAISS + HuggingFace MiniLM |
| Text-to-Speech | gTTS |
| Crisis Detection | Keyword regex |
| Context Window | Rolling 8-turn memory |

**Disclaimer:** Echo is not a licensed therapist. If you are in crisis, please contact a professional or crisis line immediately.
            """, elem_classes=["output-text"])

    gr.HTML("<div style='text-align:center;padding:16px;color:rgba(240,238,255,0.4);font-size:0.78rem'>Echo - Built with Groq + LangChain + Gradio | You are stronger than you think.</div>")

    app.load(fn=on_load, outputs=[history_state, chatbot])

print("Gradio app built successfully!")

/tmp/ipykernel_11860/4061101344.py:83: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=CUSTOM_CSS, title="Echi") as app:
/tmp/ipykernel_11860/4061101344.py:100: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(


Gradio app built successfully!


## 🚀 Cell 5 — Launch the App

In [27]:
app.launch(
    share=True,
    debug=True,
    show_error=True,


    quiet=False
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b4573068001635adf7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7861 <> https://b4573068001635adf7.gradio.live


---
## 📖 How to Use & Deploy

### ✅ Testing in Google Colab
1. Run **Cell 1** — installs packages (~2 min)
2. Run **Cell 2** — enter your Groq API key ([get one free at console.groq.com](https://console.groq.com))
3. Run **Cell 3** — loads RAG, LLM, TTS, mood, coping tools
4. Run **Cell 4** — builds the Gradio UI
5. Run **Cell 5** — launches the app. **Click the `share=True` public URL** in the output!

### 🤗 Deploying to Hugging Face Spaces
See the `DEPLOYMENT.md` file for step-by-step instructions.
---